# 029 — Procesos de decisión de Markov

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** `Q(0,a) = 0.8·(10+0.8·0) + 0.2·(0+0.8·0) = 8.0`; `Q(0,b) = 1 + 0.8·0 = 1.0`. `V₁(0) = 8.0`, política: **a**. Con V=0 solo cuenta la recompensa inmediata esperada.

**E2.** `Q(0,a) = 0.8·10 + 0.2·0.8·8 = 9.28`; `Q(0,b) = 1 + 0.8·8 = 7.4`. `V₂(0) = 9.28`. Punto fijo de la acción a: `v = 8 + 0.16v` → `v = 8/0.84 ≈ 9.524`. Verificación: en ese punto `Q(0,b) = 1 + 0.8·9.524 ≈ 8.62 < 9.524`, así que `a` sigue siendo óptima y el punto fijo es consistente. Cada iteración cierra la brecha por un factor ≈ γ·P(quedarse): la contracción de Bellman en acción.

**E3.** Empate de puntos fijos: `8/(1−0.2γ) = 1/(1−γ)` → `8(1−γ) = 1−0.2γ` → `γ = 7/7.8 ≈ 0.897`. Para `γ > 0.897` gana **b**: cobrar +1 para siempre vale `1/(1−γ)`, que crece sin cota con la paciencia, mientras que `a` paga 10 una sola vez y termina. Con nuestro `γ = 0.8` (< 0.897) el agente es lo bastante "impaciente" para preferir el premio grande e inmediato. Moraleja: el descuento no es un detalle numérico — cambia la política óptima.

**E4.** `Q(0,a) = 8 − 2 = 6` en el primer paso, `Q(0,b) = 1`: sigue ganando **a**. El costo fijo reduce el margen (punto fijo `6/0.84 ≈ 7.14`) pero no invierte la decisión mientras supere el flujo de b (`1/(1−0.8) = 5`).


In [ ]:
result = run_lab("workflow", seed=29)
assert result["kind"] == "workflow"
assert result["evidence"]
show(result)


In [ ]:
gamma = 0.8
v = 0.0
for k in range(1, 5):
    qa = 0.8*(10 + gamma*0) + 0.2*(0 + gamma*v)
    qb = 1 + gamma*v
    v = max(qa, qb)
    print(f"iter {k}: Q(a)={qa:.3f} Q(b)={qb:.3f} V={v:.3f}")
print("punto fijo accion a:", 8/(1 - 0.2*gamma))
print("punto fijo accion b:", 1/(1 - gamma))
g = 7/7.8
print(f"gamma de empate: {g:.3f} ({8/(1-0.2*g):.2f} == {1/(1-g):.2f})")


## Reflexión

1. ¿Por qué la solución de un MDP es una política y no una secuencia de acciones, a diferencia de la planificación clásica de la Parte 01?
2. En el laboratorio, ¿qué elemento juega el papel de la recompensa y cuál el de la transición estocástica? ¿Dónde está el supuesto de Markov?
3. Si γ = 0.99 en vez de 0.5, ¿qué cambia en la política resultante y en el costo de convergencia de value iteration?
